# سبق 01 - AI ایجنٹس کا تعارف

**AI ایجنٹس برائے مبتدیوں** کورس کے پہلے سبق میں خوش آمدید!

ایک **AI ایجنٹ** ایک پروگرام ہے جو ایک بڑے زبان کے ماڈل (LLM) کو اپنے استدلال انجن کے طور پر استعمال کرتا ہے اور حقیقی دنیا میں *عمل* انجام دے سکتا ہے — APIs کال کرنا، ڈیٹا بیس سے استفسار کرنا، یا کوڈ چلانا — تاکہ صارف کی جانب سے کسی مقصد کو حاصل کیا جا سکے۔

اس نوٹ بک میں آپ اپنا پہلا ایجنٹ بنائیں گے: ایک **ٹریول ایجنٹ** جو چھٹیوں کی منزلیں تجویز کرتا ہے۔ اس دوران آپ سیکھیں گے کہ:

1. **Microsoft Agent Framework** کا استعمال کرتے ہوئے Azure AI Foundry Agent Service سے کیسے کنیکٹ کیا جائے۔
2. ایجنٹ کو ایک **ٹول** دیں — ایک سادہ Python فنکشن جسے وہ کال کر سکتا ہے۔
3. ایجنٹ کو چلائیں اور اس کے جواب کا معائنہ کریں۔
4. ایجنٹ کے جواب کو ٹوکن بہ ٹوکن اسٹریمنگ کریں۔


## سیٹ اپ

اس نوٹ بک کو چلانے سے پہلے، یقینی بنائیں کہ آپ کے پاس:

1. **ایک Azure AI Foundry پروجیکٹ** ہے جس میں ایک ڈپلائے شدہ چیٹ ماڈل (مثلاً `gpt-4o-mini`) موجود ہو۔
2. **Azure CLI میں لاگ ان** کیا ہوا ہے — اپنے ٹرمینل میں `az login` چلائیں۔
3. **ضروری ماحول کے متغیرات سیٹ کیے ہوئے ہیں:**
   - `AZURE_AI_PROJECT_ENDPOINT` — آپ کے Azure AI Foundry پروجیکٹ کا اینڈپوائنٹ۔
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — آپ کے ڈپلائے شدہ ماڈل کا نام۔

نیچے والا سیل آپ کو درکار Python پیکجز انسٹال کرتا ہے۔


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## اپنا پہلا ایجنٹ بنانا

ایک ایجنٹ کو دو چیزوں کی ضرورت ہوتی ہے:

- **ہدایتیں** جو اسے بتاتی ہیں *وہ کون ہے* اور *کیسے برتاؤ کرنا ہے* (ایک سسٹم پرامپٹ)۔
- **اوزار** — Python کے فنکشنز جنہیں `@tool` سے سجا گیا ہو، جنہیں ایجنٹ معلومات حاصل کرنے یا کارروائیاں کرنے کے لیے کال کر سکتا ہے۔

نیچے ہم ایک سادہ آلہ کی تعریف کرتے ہیں جو مشہور تعطیلات کے مقامات کی فہرست واپس کرتا ہے۔ جب صارف سفر کی سفارشات طلب کرے گا تو ایجنٹ اس آلے کا استعمال کرے گا۔


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## اسٹریمنگ جوابات

زیادہ متعامل تجربے کے لیے آپ ایجنٹ کے جواب کو **اسٹریمنگ** کر سکتے ہیں۔ پورے جواب کے انتظار کرنے کی بجائے، ایجنٹ ٹیکسٹ کے حصے جنریٹ ہوتے ہی فراہم کرتا ہے۔ یہ خاص طور پر چیٹ انٹرفیسز میں مفید ہے جہاں آپ چاہتے ہیں کہ آؤٹ پٹ حقیقی وقت میں دکھایا جائے۔


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## خلاصہ

اس سبق میں آپ نے سیکھا کہ:

- **ایک پرووائڈر بنائیں** جو `AzureAIProjectAgentProvider` کے ذریعے Azure AI Foundry Agent سروس سے جُڑتا ہے۔
- **ایک ٹول کی تعریف کریں** `@tool` ڈیکوریٹر استعمال کرتے ہوئے تاکہ ایجنٹ آپ کے Python فنکشنز کو کال کر سکے۔
- **ایجنٹ چلائیں** صارف کے پیغام کے ساتھ اور اس کے جواب کو پرنٹ کریں۔
- **جوابات کو اسٹریم کریں** بر وقت آؤٹ پٹ کے لیے۔

اگلے سبق میں ہم ایجنٹک فریم ورکس کو مزید گہرائی میں دیکھیں گے اور سیکھیں گے کہ ایجنٹس کو زیادہ طاقتور ٹولز اور کثیر مرحلہ تجزیاتی صلاحیتیں کیسے دیں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**دستخطی نوٹ**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ اگرچہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم آگاہ رہیں کہ خودکار تراجم میں غلطیاں یا نقائص ہو سکتے ہیں۔ اصل دستاویز اپنی مادری زبان میں ہی مستند ماخذ سمجھی جائے۔ اہم معلومات کے لیے پیشہ ور انسانی ترجمہ تجویز کیا جاتا ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تعبیر کی ذمہ داری ہم پر نہیں ہوگی۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
